# U23 — Cloud Storage & Databases (finish): Lab

### Real-world brief: making a plant-telemetry database fast & scalable

An industrial plant streams **200,000 sensor readings** from 30 machines into a database, and the analytics queries are getting slow. In this Part-2 lab you'll apply the scaling and performance techniques from the deck **hands-on** with SQLite + Python: **indexing & query plans**, a **cache-aside** layer, **read replicas** with lag, **sharding** across nodes, a **materialized view**, and basic **security** (PII masking, read-only access).

**Resources provided:** `plant_telemetry.csv` (≈200k readings) and `machines.csv` (asset master). Everything runs locally — but each technique maps directly to its cloud equivalent.

_Phase F — Data Engineering._

#objectives

Speed up queries with indexes and read the query plan

Build a cache-aside layer and measure the hit-rate speedup

Simulate read replicas and observe replication lag

Shard data across nodes and run a scatter-gather query

Add a materialized view and basic security (masking, read-only)

#how to use this lab

Worked demos teach the pattern; 🧪 LAB EXERCISE cells are real tasks — replace `# YOUR CODE HERE`. Run top to bottom with Shift+Enter.

In [ ]:
# === SETUP: build the source files if missing ===
import os
import numpy as np
import pandas as pd


def build_plant(tele_path="plant_telemetry.csv", mach_path="machines.csv", seed=232, verbose=False):
    """A larger industrial-IoT telemetry table + machine master — sized so that indexing,
    caching, sharding and materialized-view effects are actually MEASURABLE (U23 Part 2).

      machines.csv          30 machines (asset master)
      plant_telemetry.csv   ~200k sensor readings (the high-volume table)
    """
    rng = np.random.default_rng(seed)

    N_MACH = 30
    machines = pd.DataFrame({
        "machine_id": [f"M{i:03d}" for i in range(1, N_MACH + 1)],
        "line": rng.choice(["LINE_1", "LINE_2", "LINE_3"], N_MACH),
        "machine_type": rng.choice(["pump", "compressor", "conveyor", "press"], N_MACH),
        "install_year": rng.integers(2014, 2023, N_MACH),
        "criticality": rng.choice(["low", "medium", "high"], N_MACH, p=[0.4, 0.4, 0.2]),
    })
    machines.to_csv(mach_path, index=False)

    N = 200_000
    mids = machines.machine_id.values
    metric = rng.choice(["temp", "vibration", "pressure", "current"], N, p=[0.3, 0.3, 0.2, 0.2])
    base = {"temp": 68, "vibration": 2.6, "pressure": 6.2, "current": 31}
    sd = {"temp": 7, "vibration": 0.9, "pressure": 0.7, "current": 4}
    value = np.array([rng.normal(base[m], sd[m]) for m in metric]).round(3)
    ts = pd.Timestamp("2024-06-01") + pd.to_timedelta(np.sort(rng.uniform(0, 14 * 86400, N)), unit="s")
    tele = pd.DataFrame({
        "reading_id": np.arange(N),
        "ts": ts.round("s"),
        "machine_id": rng.choice(mids, N),
        "metric": metric,
        "value": value,
        "quality_flag": rng.choice(["ok", "suspect"], N, p=[0.97, 0.03]),
    })
    tele.to_csv(tele_path, index=False)

    if verbose:
        print("machines:", machines.shape, "| telemetry:", tele.shape)
        print("distinct machines:", tele.machine_id.nunique(), "| metrics:", tele.metric.nunique())
        print("date span:", tele.ts.min(), "->", tele.ts.max())
    return machines, tele

if not (os.path.exists('plant_telemetry.csv') and os.path.exists('machines.csv')):
    build_plant(); print('Generated source files.')
else:
    print('Found the provided source files.')

In [ ]:
import pandas as pd, numpy as np, sqlite3, time, hashlib
tele = pd.read_csv('plant_telemetry.csv', parse_dates=['ts'])
mach = pd.read_csv('machines.csv')
print('telemetry:', tele.shape, '| machines:', mach.shape)
# load into a single SQLite 'database'
DB = 'plant.db'
if os.path.exists(DB): os.remove(DB)
con = sqlite3.connect(DB)
tele.assign(ts=tele.ts.astype(str)).to_sql('telemetry', con, index=False)
mach.to_sql('machines', con, index=False)
con.commit()
print('loaded into SQLite.')

def timed(sql, params=(), repeats=5):
    best = min(_t(sql, params) for _ in range(repeats)); return best
def _t(sql, params):
    t = time.perf_counter(); con.execute(sql, params).fetchall(); return (time.perf_counter()-t)*1000

#1. Indexing & query plans

In [ ]:
# -----------------------------------------------------------
# 🔹 1A. A FILTER QUERY: scan vs index
# -----------------------------------------------------------
q = "SELECT * FROM telemetry WHERE machine_id = ? AND metric = ?"
print('plan BEFORE:', con.execute('EXPLAIN QUERY PLAN ' + q, ('M007', 'temp')).fetchall()[0][-1])
t0 = timed(q, ('M007', 'temp'))
con.execute('CREATE INDEX idx_mach_metric ON telemetry(machine_id, metric)'); con.commit()
print('plan AFTER :', con.execute('EXPLAIN QUERY PLAN ' + q, ('M007', 'temp')).fetchall()[0][-1])
t1 = timed(q, ('M007', 'temp'))
print(f'query: {t0:.2f} ms (scan) -> {t1:.2f} ms (indexed) = {t0/max(t1,1e-6):.1f}x faster')

#### 🧪 EXERCISE 1 — A covering index
A *covering* index contains every column a query needs, so the database never touches the table.
1. Write a query that selects only `value` filtered by `machine_id` and `metric`. Time it with the existing index.
2. Create an index on `(machine_id, metric, value)` and re-time — the plan should say it uses the index without a table lookup.
3. In a comment, explain the trade-off (indexes speed reads but cost write time and storage).

In [ ]:
# 1-2. covering index demo + timing
# YOUR CODE HERE

# 3. index trade-off: ...   (comment)

#2. Cache-aside caching

In [ ]:
# -----------------------------------------------------------
# 🔹 2A. A CACHE-ASIDE LAYER in front of an expensive query
# -----------------------------------------------------------
def expensive_query(machine_id):
    # an aggregation the app asks for repeatedly
    sql = 'SELECT metric, AVG(value) FROM telemetry WHERE machine_id=? GROUP BY metric'
    return tuple(con.execute(sql, (machine_id,)).fetchall())

cache = {}; hits = misses = 0
def get_machine_avgs(machine_id):
    global hits, misses
    if machine_id in cache:           # 1) check cache
        hits += 1; return cache[machine_id]
    misses += 1                        # 2) miss -> hit the DB
    result = expensive_query(machine_id)
    cache[machine_id] = result         # 3) fill the cache
    return result

import random
queries = [f'M{random.randint(1,30):03d}' for _ in range(2000)]   # repeated access pattern
t = time.perf_counter()
for mid in queries: get_machine_avgs(mid)
elapsed = (time.perf_counter()-t)*1000
print(f'2000 lookups in {elapsed:.0f} ms | hits={hits} misses={misses} | hit-rate={hits/2000:.1%}')
print('Only the first lookup per machine touches the DB; the rest are served from memory.')

#### 🧪 EXERCISE 2 — Invalidation & speedup
1. Measure the average time of a **cache hit** vs a **cache miss** (a miss runs `expensive_query`). Report the speedup.
2. Cache **invalidation**: write `record_reading(machine_id, ...)` that inserts a row AND evicts that machine's cache entry (so the next read recomputes). Show the entry is gone after a write.
3. In a comment, explain why stale caches are the classic caching bug, and what a TTL would add.

In [ ]:
# 1-2. hit vs miss timing; write-with-invalidation
# YOUR CODE HERE

# 3. cache staleness & TTL: ...   (comment)

#3. Read replicas & replication lag

In [ ]:
# -----------------------------------------------------------
# 🔹 3A. PRIMARY takes writes; a REPLICA serves reads (with lag)
# -----------------------------------------------------------
primary = sqlite3.connect('primary.db'); primary.execute('DROP TABLE IF EXISTS kv')
primary.execute('CREATE TABLE kv (machine_id TEXT PRIMARY KEY, status TEXT)')
primary.execute("INSERT INTO kv VALUES ('M007','ok')"); primary.commit()

def make_replica(src='primary.db', dst='replica.db'):
    import shutil; shutil.copy(src, dst); return sqlite3.connect(dst)
replica = make_replica()   # snapshot copy = a (lagging) read replica

# a WRITE goes to the primary only
primary.execute("UPDATE kv SET status='fault' WHERE machine_id='M007'"); primary.commit()
p = primary.execute("SELECT status FROM kv WHERE machine_id='M007'").fetchone()[0]
r = replica.execute("SELECT status FROM kv WHERE machine_id='M007'").fetchone()[0]
print(f'read from PRIMARY: {p}  |  read from REPLICA: {r}  <- stale (replication lag)')
replica = make_replica()   # replication catches up (re-sync)
r2 = replica.execute("SELECT status FROM kv WHERE machine_id='M007'").fetchone()[0]
print(f'after replica re-syncs: {r2}  (eventual consistency)')

#### 🧪 EXERCISE 3 — Read/write routing
1. Write a tiny router: `def query(sql, write=False)` that sends writes to `primary` and reads to `replica`. Use it to do one write then one read.
2. In a comment, explain when routing reads to replicas is safe (analytics, dashboards) and when it is dangerous (read-after-write where the user expects to see their own change immediately).

In [ ]:
# 1. read/write router
# YOUR CODE HERE

# 2. when replica reads are (un)safe: ...   (comment)

#4. Sharding across nodes

In [ ]:
# -----------------------------------------------------------
# 🔹 4A. SHARD the telemetry across N nodes by machine_id
# -----------------------------------------------------------
import zlib
N_SHARDS = 4
def shard_of(machine_id): return zlib.crc32(machine_id.encode()) % N_SHARDS

shards = [sqlite3.connect(f'shard_{i}.db') for i in range(N_SHARDS)]
for sc in shards:
    sc.execute('DROP TABLE IF EXISTS telemetry')
    sc.execute('CREATE TABLE telemetry (machine_id TEXT, metric TEXT, value REAL)')
rows = con.execute('SELECT machine_id, metric, value FROM telemetry').fetchall()
for mid, metric, val in rows:
    shards[shard_of(mid)].execute('INSERT INTO telemetry VALUES (?,?,?)', (mid, metric, val))
for sc in shards: sc.commit()
sizes = [sc.execute('SELECT COUNT(*) FROM telemetry').fetchone()[0] for sc in shards]
print('rows per shard:', sizes, '-> total', sum(sizes))

# a single-machine query hits ONE shard (routed); a global query is scatter-gather
one = shards[shard_of('M007')].execute("SELECT COUNT(*) FROM telemetry WHERE machine_id='M007'").fetchone()[0]
print('M007 lives on shard', shard_of('M007'), 'with', one, 'rows (routed read, no fan-out)')

#### 🧪 EXERCISE 4 — Scatter-gather & skew
1. Compute the **global** average `value` per metric by querying **every** shard and combining the partial results (scatter-gather). Compare to the single-DB answer.
2. Show the downside of a bad shard key: count rows if you sharded by **metric** (only 4 distinct values) instead of machine_id — note how few shards would be used and why that causes hotspots.
3. In a comment, state the rule for picking a shard key (high cardinality, even access).

In [ ]:
# 1-2. scatter-gather global average; metric-skew demonstration
# YOUR CODE HERE

# 3. shard-key rule: ...   (comment)

#5. Materialized view & security

In [ ]:
# -----------------------------------------------------------
# 🔹 5A. MATERIALIZED VIEW: precompute an expensive rollup
# -----------------------------------------------------------
rollup_sql = '''SELECT machine_id, metric, AVG(value) avg_v, COUNT(*) n
                FROM telemetry GROUP BY machine_id, metric'''
t_live = timed(rollup_sql)                       # computed on the fly every time
con.execute('DROP TABLE IF EXISTS mv_machine_metric')
con.execute('CREATE TABLE mv_machine_metric AS ' + rollup_sql); con.commit()
con.execute('CREATE INDEX idx_mv ON mv_machine_metric(machine_id)'); con.commit()
t_mv = timed('SELECT * FROM mv_machine_metric WHERE machine_id=?', ('M007',))
print(f'aggregation live: {t_live:.2f} ms  |  read from materialized view: {t_mv:.2f} ms')
print('Trade-off: the view must be REFRESHED when underlying data changes (it can go stale).')

In [ ]:
# -----------------------------------------------------------
# 🔹 5B. SECURITY: mask PII-like fields & a read-only connection
# -----------------------------------------------------------
# tokenize an identifier so analysts can join without seeing the raw id
def tokenize(s, salt='plant2024'):
    return hashlib.sha256((salt + str(s)).encode()).hexdigest()[:12]
print('M007 tokenized ->', tokenize('M007'))
# a read-only connection: writes are rejected (least privilege)
ro = sqlite3.connect('file:plant.db?mode=ro', uri=True)
try:
    ro.execute('DELETE FROM telemetry'); print('write succeeded (unexpected!)')
except Exception as e:
    print('write blocked on read-only connection:', type(e).__name__)

#### 🧪 EXERCISE 5 — Keyset pagination
Offset pagination (`LIMIT n OFFSET k`) gets slower as `k` grows because the DB still scans the skipped rows.
1. Time `... ORDER BY reading_id LIMIT 50 OFFSET 150000` vs **keyset** pagination `... WHERE reading_id > 150000 ORDER BY reading_id LIMIT 50`.
2. In a comment, explain why keyset pagination stays fast on deep pages (it seeks via the index instead of counting past skipped rows).

In [ ]:
# 1. offset vs keyset pagination timing
# YOUR CODE HERE

# 2. why keyset wins on deep pages: ...   (comment)

#📘 Summary

| Technique | What you measured |
| --------- | ----------------- |
| Indexing | scan → index search, large speedup |
| Cache-aside | repeated reads served from memory |
| Read replicas | offload reads; accept replication lag |
| Sharding | spread data by key; route or scatter-gather |
| Materialized view | precompute rollups; refresh to stay fresh |
| Security | tokenize PII; read-only least-privilege access |

**Core lesson:** production performance comes from a toolbox of trade-offs — cache hot reads, index what you filter, replicate and shard to scale out, precompute expensive rollups — each buying speed or scale at the cost of freshness, complexity or write overhead.

**Next — U24:** apply this same engineering discipline to machine-learning systems (MLOps).